In [1]:
import torch
from transformers import pipeline
import os

class CodeGeneratorModel:
    def __init__(self, model_name, task="text-generation", torch_dtype=torch.float16, device_map="auto"):
        print(f"Loading model '{model_name}' to T4 GPU... (this takes a minute)")
        self.generator = pipeline(
            model=model_name,
            task=task,
            torch_dtype=torch_dtype,
            device_map=device_map,
        )
        print("Model loaded successfully!")
        self.tokenizer = self.generator.tokenizer

    def generate(self, instruction, max_new_tokens=300):
        prompt_template = """### Instruction
{instruction}
### Response
"""
        prompt = prompt_template.format(instruction=instruction)
        result = self.generator(
            prompt,
            max_new_tokens=max_new_tokens,
            return_full_text=False,
            pad_token_id=self.tokenizer.eos_token_id
        )
        return result[0]["generated_text"].strip()

d:\LLM_Reasoning_Research\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class OutputManager:
    def __init__(self, output_base_dir="./model_outputs"):
        self.output_base_dir = output_base_dir
        os.makedirs(self.output_base_dir, exist_ok=True)
        print(f"Output directory created at: {os.path.abspath(self.output_base_dir)}")

    def save_output(self, test_id, instruction, model_response):
        test_dir = os.path.join(self.output_base_dir, f"test_{test_id}")
        os.makedirs(test_dir, exist_ok=True)

        prompt_file = os.path.join(test_dir, "instruction.txt")
        response_file = os.path.join(test_dir, "model_response.txt")

        with open(prompt_file, "w", encoding="utf-8") as f:
            f.write(instruction)

        with open(response_file, "w", encoding="utf-8") as f:
            f.write(model_response)

        print(f"Saved output for Test {test_id} to {test_dir}")

In [ ]:
class TestManager:
    def __init__(self, model, output_manager, test_instructions):
        self.model = model
        self.output_manager = output_manager
        self.test_instructions = test_instructions

    def run_tests(self):
        print("\nModel loaded successfully! Starting tests...\n")
        for i, instruction in enumerate(self.test_instructions, 1):
            print(f"--- [ Test {i} of {len(self.test_instructions)} ] ---")
            print(f"USER PROMPT: {instruction}\n")

            model_response = self.model.generate(instruction)

            print("MODEL RESPONSE:")
            print(model_response)
            print("\n" + "=" * 70 + "\n")

            self.output_manager.save_output(i, instruction, model_response)
        print("All tests completed and outputs saved!")

In [ ]:
PROMPT_TEMPLATE = """### Instruction
{instruction}
### Response
"""

test_instructions = [
    # Test 1: Regular Expressions
    "Write a Python function to check if a given string is a valid email address using the `re` module.",

    # Test 2: SQL Generation
    "Write a SQL query to find the names and ages of all employees in the 'Engineering' department who earn a salary greater than 80000.",

    # Test 3: Data Manipulation (Pandas)
    "Write a Python snippet using pandas to read 'data.csv', fill any missing values in the 'price' column with the mean of that column, and save it to 'cleaned_data.csv'.",

    # Test 4: Concept Explanation
    "Explain the primary differences between a List and a Tuple in Python, and provide a brief example of when you would choose one over the other."
]

# Initialize the CodeGeneratorModel
generator_model = CodeGeneratorModel(
    model_name="TechxGenus/starcoder2-3b-instruct",
    torch_dtype=torch.float16,
    device_map="auto",
)

# Initialize the OutputManager
output_manager = OutputManager(output_base_dir="./model_outputs")

# Initialize and run the TestManager
test_manager = TestManager(generator_model, output_manager, test_instructions)
test_manager.run_tests()

### Loading the APPS Dataset for Evaluation

Now, let's load the `codeparrot/apps` dataset, which consists of programming problems. We'll use a subset (introductory level) for a baseline evaluation, as the full dataset is quite large. This dataset will provide the questions for our code generation model.